In [1]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
import random

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0731 15:10:50.328000 76927 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
language_classifier = pipeline(
    "text-classification",
    model = "papluca/xlm-roberta-base-language-detection"
)

def lang_result(text):
    results = language_classifier(
        text,
        truncation=True
    )
    return results[0]["label"]

theme_classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    multi_label = True
)

def theme_result(text, theme_labels):
    return theme_classifier(
        text,
        candidate_labels=theme_labels,
        hypothesis_template="This post discusses {}.",
        multi_label=True
    )

model_name = "yangheng/deberta-v3-base-absa-v1.1"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


def aspect_sentiment(text, aspect, batch_size=16, max_length=512, stride=64):
    encoded = tokenizer(
        text,
        aspect,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        padding=True,
        return_tensors="pt"
    )

    input_keys = ["input_ids", "attention_mask", "token_type_ids"]
    input_keys = [k for k in input_keys if k in encoded]

    all_probs = []

    with torch.inference_mode():
        n_chunks = encoded["input_ids"].shape[0]

        for start in range(0, n_chunks, batch_size):
            end = start + batch_size

            batch = {
                k: encoded[k][start:end].to(device)
                for k in input_keys
            }

            outputs = model(**batch)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs)

    avg_probs = torch.cat(all_probs, dim=0).mean(dim=0).cpu()

    return {
        model.config.id2label[i]: float(avg_probs[i])
        for i in range(len(avg_probs))
    }

Device set to use mps:0
Device set to use mps:0


Using device: mps


/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [3]:
def remove_links(text):
    url_pattern = re.compile(r'[\[(]?(?:https?://|www\.)\S+[\])]?' )
    return url_pattern.sub('', text)

In [4]:
BASE_DIR = Path.cwd()
file_path = (
    BASE_DIR
    / "brightdata_social_exports"
    / "reddit_updated_datacenters_posts_sample_300.json"
)
with file_path.open("r", encoding="utf-8") as f:
    reddit_posts = json.load(f)

In [5]:
english_post_ids = []
a = 0

for r_post in reddit_posts:
    unclean_text = r_post["description"]
    text = remove_links(unclean_text)
    language = lang_result(text)
    pid = r_post["post_id"]
    if language == 'en':
        english_post_ids.append(pid)
    a += 1
    print("Posts scanned:", a, end="\r")

In [9]:
print(len(english_post_ids))

284


In [ ]:
all_post_ids = []

tax_post_ids = []
env_post_sentiments = []
env_post_sentiment_degrees = []

data_security_post_ids = []
infr_post_sentiments = []
infr_post_sentiment_degrees = []

housing_post_ids = []
housing_post_sentiments = []
housing_post_sentiment_degrees = []

econ_post_ids = []
econ_post_sentiments = []
econ_post_sentiment_degrees = []

life_qual_post_ids = []
life_qual_post_sentiments = []
life_qual_post_sentiment_degrees = []

aesth_post_ids = []
aesth_post_sentiments = []
aesth_post_sentiment_degrees = []

gov_post_ids = []
gov_post_sentiments = []
gov_post_sentiment_degrees = []

tech_post_ids = []
tech_post_sentiments = []
tech_post_sentiment_degrees = []

not_useful_post_ids = []

themes = ["visual impact of datacenters", "infrastructure and house utilities", "housing costs and property values", "economy and jobs", "quality of life, noise, and light pollution", "environmental impact", "government decisions and policies", "technology performance and growth"]

matched_posts = 0
more_than_one_theme_posts = 0

a = 0

for r_post in reddit_posts:

    post_id = r_post["post_id"]
    if post_id not in english_post_ids:
        continue
    all_post_ids.append(post_id)
    unclean_text = r_post["title"] + " " + r_post["description"]
    text = remove_links(unclean_text)


    final_labels = []

    theme_scores = theme_result(
        text,
        themes
    )
    
    for i in range(len(theme_scores['labels'])):
        if theme_scores['scores'][i] > 0.5:
            final_labels.append(theme_scores['labels'][i])
    
    if len(final_labels) > 0:
        matched_posts += 1
    else:
        not_useful_post_ids.append(post_id)
    
    if len(final_labels) > 1:
        more_than_one_theme_posts += 1
    

    for label in final_labels:
        total_sentiment = aspect_sentiment(text, label)
        post_sentiment = max(total_sentiment, key=total_sentiment.get)
        post_degree = max(total_sentiment.values())

        if label == "visual impact of datacenters":
            aesth_post_ids.append(post_id)
            aesth_post_sentiments.append(post_sentiment)
            aesth_post_sentiment_degrees.append(post_degree)
        if label == "infrastructure and house utilities":
            infr_post_ids.append(post_id)
            infr_post_sentiments.append(post_sentiment)
            infr_post_sentiment_degrees.append(post_degree)
        if label == "housing costs and property values":
            housing_post_ids.append(post_id)
            housing_post_sentiments.append(post_sentiment)
            housing_post_sentiment_degrees.append(post_degree)
        if label == "economy and jobs":
            econ_post_ids.append(post_id)
            econ_post_sentiments.append(post_sentiment)
            econ_post_sentiment_degrees.append(post_degree)
        if label == "quality of life, noise, and light pollution":
            life_qual_post_ids.append(post_id)
            life_qual_post_sentiments.append(post_sentiment)
            life_qual_post_sentiment_degrees.append(post_degree)
        if label == "environmental impact":
            env_post_ids.append(post_id)
            env_post_sentiments.append(post_sentiment)
            env_post_sentiment_degrees.append(post_degree)
        if label == "government decisions and policies":
            gov_post_ids.append(post_id)
            gov_post_sentiments.append(post_sentiment)
            gov_post_sentiment_degrees.append(post_degree)
        if label == "technology performance and growth":
            tech_post_ids.append(post_id)
            tech_post_sentiments.append(post_sentiment)
            tech_post_sentiment_degrees.append(post_degree)

        a += 1
        print("Posts scanned:", a, end="\r")

print("Total posts scanned:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))
print("Found technological posts:", len(tech_post_ids))
print(not_useful_post_ids)

print(gov_post_sentiments)
print(gov_post_sentiment_degrees)

Total posts scanned: 284
Total posts with a theme: 204
Found environmental posts: 10
Found infrastructure posts: 5
Found housing posts: 1
Found economic posts: 14
Found life quality posts: 2
Found aesthetic posts: 2
Found government posts: 33
Found technological posts: 162
['t3_1niebsx', 't3_1bpxg58', 't3_1rp11ee', 't3_1t4dkf7', 't3_1tu11vu', 't3_1ruwbvk', 't3_1rsyi41', 't3_1v8w8fg', 't3_1fzzern', 't3_1sht49n', 't3_1mnq2pc', 't3_1s07ww4', 't3_1t6mbls', 't3_1s1pcup', 't3_1o30c5o', 't3_1twuyap', 't3_1ti8qcd', 't3_1ssipkp', 't3_1tm9ekc', 't3_1tv0l4g', 't3_1mxp1ot', 't3_1ulxy95', 't3_1rzmda2', 't3_1szk3gs', 't3_1mkvbo1', 't3_1l94skv', 't3_1szo4ht', 't3_1qto367', 't3_1pfygro', 't3_1q8o58q', 't3_1mk78ey', 't3_1pdg66b', 't3_1ovaeoz', 't3_1se89cp', 't3_1qc8n2l', 't3_1q3d9zm', 't3_1qdpbap', 't3_1ttk3e7', 't3_1t1bw0p', 't3_1jrb1mm', 't3_1eay6ya', 't3_yrizid', 't3_1v7ev63', 't3_1rnhxdh', 't3_1r2ziuz', 't3_1tcl3f4', 't3_1gbt31o', 't3_1tc5llt', 't3_1mtt6h0', 't3_1r2gp4b', 't3_1v0pevi', 't3_1rgp11a'

In [11]:
posts_by_id = {post["post_id"]: post for post in reddit_posts}
gov_links = []

for id in gov_post_ids:
    post = posts_by_id.get(id)
    gov_links.append(post["url"])

print(gov_links)

['https://www.reddit.com/r/SouthJersey/comments/1v765o7/ai_is_being_shutout_all_over_south_jersey_but/', 'https://www.reddit.com/r/ChainIntel_SolarFarm/comments/1twoc9z/why_is_americas_climate_capital_still_39_gas/', 'https://www.reddit.com/r/NorthCarolina/comments/1v5kfd3/data_center_consultant_lobbies_greensboro_city/', 'https://www.reddit.com/r/NIOCORP_MINE/comments/1ozsu1u/niocorpchina_deal_buys_us_time_to_build_critical/', 'https://www.reddit.com/r/Martinsburg/comments/1rjtkgt/data_centers_report_on_mondays_planning', 'https://www.reddit.com/user/TopherLeritz/comments/1tn10ob/some_other_major_issues/', 'https://www.reddit.com/r/UnderReportedNews/comments/1udha9w/fire_department_turns_down_250000_google_donation/', 'https://www.reddit.com/r/CanadaStocks/comments/1tn8ggp/canada_keeps_treating_mining_more_like_strategic/', 'https://www.reddit.com/r/Virginia/comments/1sd5af2/google_struck_a_secret_deal_for_3_data_centers_in/', 'https://www.reddit.com/r/companysetupbh/comments/1nld9gh/

In [9]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids, tech_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

posts_by_id = {post["post_id"]: post for post in reddit_posts}

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology", "AWS", "Amazon", "Google", "Microsoft", "Azure", "Meta", "Oracle", "Equinix", "Digital Realty", "IBM", "Facebook", "Apple", "QTS", "Vantage", "CyrusOne", "CoreSite"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_num_comments = []
    post_upvotes = []
    post_subreddits = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        unclean_text = post["title"] + " " + post["description"]
        text = remove_links(unclean_text)
        post_texts.append(text)
        post_dates.append(post["date_posted"])
        post_num_comments.append(post["num_comments"])
        post_upvotes.append(post["num_upvotes"])
        post_subreddits.append(post["community_name"])

    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["number of comments"] = post_num_comments
    df1["upvotes"] = post_upvotes
    df1["subreddit"] = post_subreddits

    
    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]:
        df1[col] = False

    for col in ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]:
        df1[col] = None

    if theme == env_post_ids:
        df1["environment"] = True
        df1["environment sentiment"] = env_post_sentiments
        df1["environment sentiment degree"] = env_post_sentiment_degrees
    if theme == infr_post_ids:
        df1["infrastructure"] = True
        df1["infrastructure sentiment"] = infr_post_sentiments
        df1["infrastructure sentiment degree"] = infr_post_sentiment_degrees
    if theme == housing_post_ids:
        df1["housing"] = True
        df1["housing sentiment"] = housing_post_sentiments
        df1["housing sentiment degree"] = housing_post_sentiment_degrees
    if theme == econ_post_ids:
        df1["economy"] = True
        df1["economy sentiment"] = econ_post_sentiments
        df1["economy sentiment degree"] = econ_post_sentiment_degrees
    if theme == life_qual_post_ids:
        df1["life quality"] = True
        df1["life quality sentiment"] = life_qual_post_sentiments
        df1["life quality sentiment degree"] = life_qual_post_sentiment_degrees
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
        df1["aesthetics sentiment"] = aesth_post_sentiments
        df1["aesthetics sentiment degree"] = aesth_post_sentiment_degrees
    if theme == gov_post_ids:
        df1["government"] = True
        df1["government sentiment"] = gov_post_sentiments
        df1["government sentiment degree"] = gov_post_sentiment_degrees
    if theme == tech_post_ids:
        df1["technology"] = True
        df1["technology sentiment"] = tech_post_sentiments
        df1["technology sentiment degree"] = tech_post_sentiment_degrees
    posts = pd.concat([posts, df1], ignore_index=True)

/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77712/4274511221.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77712/4274511221.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  posts = pd.concat([posts, df1], ignore_index=True)
/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_77712/4274511221.py:74: FutureWarning: The behavior of DataFrame concatenation wi

In [17]:
len(posts)

62

In [18]:
posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "upvotes": "int64",
    "number of comments": "int64",
    "subreddit": "string",
})

In [19]:
grouping_cols = ["ids", "text", "date", "upvotes", "number of comments", "subreddit"]

theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government", "technology"]
sent_cols  = ["environment sentiment", "environment sentiment degree", "infrastructure sentiment", "infrastructure sentiment degree", "housing sentiment", "housing sentiment degree", "economy sentiment", "economy sentiment degree", "life quality sentiment", "life quality sentiment degree", "aesthetics sentiment", "aesthetics sentiment degree", "government sentiment", "government sentiment degree", "technology sentiment", "technology sentiment degree"]

def first_non_null(s):
    return s.dropna().iloc[0] if s.notna().any() else np.nan

agg = {c: "max" for c in theme_cols}              # True if any True
agg.update({c: first_non_null for c in sent_cols}) # keep the real sentiment if present

posts = posts.groupby(grouping_cols, as_index=False, dropna=False).agg(agg)

In [20]:
len(posts)

57

In [ ]:
posts.to_json('reddit_ABSA_entire_dataframe.json', orient='records', indent=4)

In [21]:
posts.head(30)

,ids,text,date,upvotes,number of comments,subreddit,environment,infrastructure,housing,economy,...,economy sentiment,economy sentiment degree,life quality sentiment,life quality sentiment degree,aesthetics sentiment,aesthetics sentiment degree,government sentiment,government sentiment degree,technology sentiment,technology sentiment degree
0,t3_118og74,The Internet Computer - Scam or Legit? What do...,2023-02-22T04:07:17.216Z,60,238,CryptoCurrency,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.774393
1,t3_12ebj68,Something better (more reliable) than Intel NU...,2023-04-07T05:50:41.151Z,14,48,homelab,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.524171
2,t3_15zc676,SOS: Data Centres Are Running Out of Power Int...,2023-08-23T18:26:47.853Z,17,16,datacenter,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.594372
3,t3_19aay8p,Moving from info sec eng to solutions architec...,2024-01-19T04:52:08.270Z,1,1,ITCareerQuestions,False,False,False,True,...,Positive,0.603580,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.700479
4,t3_1fz8zrm,0MNNWJ Dell 500-Watts Power Supply for EMC DD1...,2024-10-08T19:56:58.446Z,1,0,ETechBuy,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.939708
5,t3_1hmnnwg,Built a Powerful and Silent AMD EPYC Home Serv...,2024-12-26T12:46:30.702Z,346,102,homelab,False,True,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.866535
6,t3_1jdl5ek,12 Tentative Ideas for US AI Policy by Luke Mu...,2025-03-17T19:18:22.898Z,8,9,slatestarcodex,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,Negative,0.416761,NaN,NaN
7,t3_1kp6tq0,Indianapolis residents push back against data ...,2025-05-17T23:56:07.061Z,394,52,Indiana,False,False,False,True,...,Neutral,0.622483,NaN,NaN,NaN,NaN,Neutral,0.593339,NaN,NaN
8,t3_1kr6jty,Microsoft Discovery- AI Agents Go From Idea to...,2025-05-20T14:40:27.865Z,4,3,OpenAI,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.967848
9,t3_1leqjk6,New office move - Equipment suggestions Moving...,2025-06-18T19:42:11.531Z,2,1,sysadmin,False,False,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Positive,0.685925


In [22]:
print(posts.columns)

Index(['ids', 'text', 'date', 'upvotes', 'number of comments', 'subreddit',
       'environment', 'infrastructure', 'housing', 'economy', 'life quality',
       'aesthetics', 'government', 'technology', 'environment sentiment',
       'environment sentiment degree', 'infrastructure sentiment',
       'infrastructure sentiment degree', 'housing sentiment',
       'housing sentiment degree', 'economy sentiment',
       'economy sentiment degree', 'life quality sentiment',
       'life quality sentiment degree', 'aesthetics sentiment',
       'aesthetics sentiment degree', 'government sentiment',
       'government sentiment degree', 'technology sentiment',
       'technology sentiment degree'],
      dtype='object')


In [23]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    if len(theme_posts) > 0:
        pos = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Positive", f"{theme} sentiment degree"].sum()
        neg = theme_posts.loc[theme_posts[f"{theme} sentiment"] == "Negative", f"{theme} sentiment degree"].sum()
        total = theme_posts[f"{theme} sentiment degree"].sum()
        return len(theme_posts), (pos-neg)/total
    else:
        return 0, None


print("Number of environmental posts: ", avg_sentiment_calculation("environment")[0], ", Average sentiment of environmental posts: ", avg_sentiment_calculation("environment")[1], sep="")
print("Number of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[0], ", Average sentiment of infrastructural posts: ", avg_sentiment_calculation("infrastructure")[1], sep="")
print("Number of housing-related posts: ", avg_sentiment_calculation("housing")[0], ", Average sentiment of housing-related posts: ", avg_sentiment_calculation("housing")[1], sep="")
print("Number of economic posts: ", avg_sentiment_calculation("economy")[0], ", Average sentiment of economic posts: ", avg_sentiment_calculation("economy")[1], sep="")
print("Number of life-quality-related posts: ", avg_sentiment_calculation("life quality")[0], ", Average sentiment of life-quality-related posts: ", avg_sentiment_calculation("life quality")[1], sep="")
print("Number of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[0], ", Average sentiment of aesthetics-related posts: ", avg_sentiment_calculation("aesthetics")[1], sep="")
print("Number of governmental posts: ", avg_sentiment_calculation("government")[0], ", Average sentiment of governmental posts: ", avg_sentiment_calculation("government")[1], sep="")
print("Number of technological posts: ", avg_sentiment_calculation("technology")[0], ", Average sentiment of technological posts: ", avg_sentiment_calculation("technology")[1], sep="")


Number of environmental posts: 1, Average sentiment of environmental posts: -1.0
Number of infrastructural posts: 1, Average sentiment of infrastructural posts: 1.0
Number of housing-related posts: 0, Average sentiment of housing-related posts: None
Number of economic posts: 5, Average sentiment of economic posts: 0.6002409547432044
Number of life-quality-related posts: 0, Average sentiment of life-quality-related posts: None
Number of aesthetics-related posts: 0, Average sentiment of aesthetics-related posts: None
Number of governmental posts: 7, Average sentiment of governmental posts: -0.28835136997941074
Number of technological posts: 48, Average sentiment of technological posts: 0.4016879260132845


In [19]:
posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}

posts_2010 = year_datasets[2010]
posts_2020 = year_datasets[2020]